In [ ]:
# Colab preprocessing for extracted BraTS test set with masks
import os
import json
import zipfile
from pathlib import Path
import numpy as np
from scipy import ndimage, interpolate
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

# =========================
# CONFIG
# =========================
ZIP_PATH = Path('/content/drive/MyDrive/symAD-ECNN/data/brats_test_with_masks_raw.zip')
WORK_DIR = Path('/content/test_eval_data')
RAW_DIR = WORK_DIR / 'raw'
PROCESSED_DIR = WORK_DIR / 'processed'
RAW_IMAGES = RAW_DIR / 'images'
RAW_MASKS = RAW_DIR / 'masks'
RAW_LABELS = RAW_DIR / 'labels'
PROCESSED_IMAGES = PROCESSED_DIR / 'images'
PROCESSED_MASKS = PROCESSED_DIR / 'masks'
PROCESSED_LABELS = PROCESSED_DIR / 'labels'
PROCESSED_META = PROCESSED_DIR / 'metadata.json'
PROCESSED_ZIP = Path('/content/drive/MyDrive/symAD-ECNN/data/brats_test_with_masks_processed.zip')

# If IXI train exists, landmarks can be learned dynamically; otherwise fallback constants
IXI_TRAIN_DIR = Path('/content/drive/MyDrive/symAD-ECNN/data/ixi_t1/processed_ixi/train')
USE_DYNAMIC_LANDMARKS = True
PERCENTILES = [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 99]
FALLBACK_LANDMARKS = np.array([
    1.11170489e-41, 5.18809652e-25, 3.05301042e-17, 4.70776470e-12,
    6.94896974e-07, 5.00224718e-03, 1.75081036e-01, 5.40189319e-01,
    7.13654902e-01, 8.37299432e-01, 9.57721521e-01
], dtype=np.float32)

# =========================
# HELPERS
# =========================
def normalize01(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-6:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def center_image_and_mask(img: np.ndarray, mask: np.ndarray, target_center=(64, 64)):
    if np.count_nonzero(img) < 100:
        return img, mask

    cy, cx = ndimage.center_of_mass(img > 0.1)
    if np.isnan(cy) or np.isnan(cx):
        return img, mask

    shift_y = target_center[0] - cy
    shift_x = target_center[1] - cx

    centered_img = ndimage.shift(img, [shift_y, shift_x], order=1, mode='constant', cval=0.0)
    centered_mask = ndimage.shift(mask.astype(np.float32), [shift_y, shift_x], order=0, mode='constant', cval=0.0)
    centered_mask = (centered_mask > 0.5).astype(np.uint8)
    return centered_img.astype(np.float32), centered_mask

def train_landmarks_from_ixi(train_dir: Path, max_samples: int = 2000) -> np.ndarray:
    files = sorted(train_dir.glob('*.npy'))
    if len(files) == 0:
        raise RuntimeError(f'No IXI files found in {train_dir}')

    rng = np.random.default_rng(42)
    sample_n = min(max_samples, len(files))
    sample_idx = rng.choice(len(files), sample_n, replace=False)

    all_landmarks = []
    for i in tqdm(sample_idx, desc='Learning landmarks'):
        arr = np.load(files[int(i)]).astype(np.float32)
        mask = arr > 0
        if mask.sum() < 100:
            continue
        all_landmarks.append(np.percentile(arr[mask], PERCENTILES))

    if len(all_landmarks) == 0:
        raise RuntimeError('Could not compute landmarks from IXI samples.')

    return np.mean(np.asarray(all_landmarks), axis=0).astype(np.float32)

def apply_nyul(img: np.ndarray, standard_landmarks: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    mask = img > 0
    if mask.sum() < 100:
        return img

    current_landmarks = np.percentile(img[mask], PERCENTILES).astype(np.float32)
    x = np.concatenate(([0.0], current_landmarks, [float(img.max())]))
    y = np.concatenate(([0.0], standard_landmarks, [float(standard_landmarks[-1])]))

    x_unique, idx = np.unique(x, return_index=True)
    y_unique = y[idx]
    if x_unique.size < 2:
        return img

    f = interpolate.interp1d(x_unique, y_unique, bounds_error=False, fill_value='extrapolate')
    out = f(img).astype(np.float32)
    out[~mask] = 0.0
    return np.clip(out, 0.0, 1.0)

# =========================
# UNZIP
# =========================
if not ZIP_PATH.exists():
    raise FileNotFoundError(f'ZIP not found: {ZIP_PATH}')

for d in [RAW_DIR, PROCESSED_DIR]:
    if d.exists():
        import shutil
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

print(f'Extracting {ZIP_PATH.name} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(RAW_DIR)

for p in [RAW_IMAGES, RAW_MASKS, RAW_LABELS, PROCESSED_IMAGES, PROCESSED_MASKS, PROCESSED_LABELS]:
    p.mkdir(parents=True, exist_ok=True)

image_files = sorted([f for f in RAW_IMAGES.glob('*.npy')])
mask_files = sorted([f for f in RAW_MASKS.glob('*.npy')])
label_files = sorted([f for f in RAW_LABELS.glob('*.npy')])
print(f'Raw images: {len(image_files)} | masks: {len(mask_files)} | labels: {len(label_files)}')

if len(image_files) == 0 or len(mask_files) == 0:
    raise RuntimeError('Missing images or masks after extraction.')

# =========================
# LANDMARKS
# =========================
if USE_DYNAMIC_LANDMARKS and IXI_TRAIN_DIR.exists():
    try:
        print('Learning Nyul landmarks from IXI train...')
        STANDARD_LANDMARKS = train_landmarks_from_ixi(IXI_TRAIN_DIR, max_samples=2000)
        print('Dynamic landmarks learned.')
    except Exception as e:
        print(f'Landmark learning failed ({e}); using fallback landmarks.')
        STANDARD_LANDMARKS = FALLBACK_LANDMARKS.copy()
else:
    STANDARD_LANDMARKS = FALLBACK_LANDMARKS.copy()
    print('Using fallback Nyul landmarks.')

# =========================
# PROCESS LOOP
# =========================
processed_rows = []

for img_fp in tqdm(image_files, desc='Preprocessing test slices'):
    base = img_fp.stem  # e.g., slice_000001
    mask_fp = RAW_MASKS / f'{base}.npy'
    label_fp = RAW_LABELS / f"label_{base.split('_')[-1]}.npy"

    if not mask_fp.exists():
        continue

    img = np.load(img_fp).astype(np.float32)
    mask = np.load(mask_fp).astype(np.uint8)
    if img.shape != (128, 128):
        from skimage.transform import resize
        img = resize(img, (128, 128), order=3, mode='reflect', anti_aliasing=True, preserve_range=True).astype(np.float32)
    if mask.shape != (128, 128):
        from skimage.transform import resize
        mask = (resize(mask, (128, 128), order=0, preserve_range=True) > 0).astype(np.uint8)

    # Center image and mask together (same transform)
    img_c, mask_c = center_image_and_mask(img, mask, target_center=(64, 64))

    # Nyul on image only
    img_n = apply_nyul(img_c, STANDARD_LANDMARKS).astype(np.float32)

    # Label from mask (robust)
    label = np.array([1 if int(mask_c.sum()) > 0 else 0], dtype=np.uint8)

    np.save(PROCESSED_IMAGES / f'{base}.npy', img_n)
    np.save(PROCESSED_MASKS / f'{base}.npy', mask_c)
    np.save(PROCESSED_LABELS / f"label_{base.split('_')[-1]}.npy", label)

    processed_rows.append({
        'slice_id': base,
        'label': int(label[0]),
        'mask_pixels': int(mask_c.sum())
    })

with open(PROCESSED_META, 'w', encoding='utf-8') as f:
    json.dump({
        'n_samples': len(processed_rows),
        'landmarks': STANDARD_LANDMARKS.tolist(),
        'rows': processed_rows[:20]
    }, f, indent=2)

# Optional: zip processed set for reuse
if PROCESSED_ZIP.exists():
    PROCESSED_ZIP.unlink()
with zipfile.ZipFile(PROCESSED_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for sub in [PROCESSED_IMAGES, PROCESSED_MASKS, PROCESSED_LABELS]:
        for fp in sorted(sub.glob('*.npy')):
            zf.write(fp, arcname=str(fp.relative_to(PROCESSED_DIR)).replace('\\', '/'))
    zf.write(PROCESSED_META, arcname='metadata.json')

print('\n=== PREPROCESSING COMPLETE ===')
print(f'Processed images: {len(list(PROCESSED_IMAGES.glob("*.npy"))):,}')
print(f'Processed masks: {len(list(PROCESSED_MASKS.glob("*.npy"))):,}')
print(f'Processed labels: {len(list(PROCESSED_LABELS.glob("*.npy"))):,}')
print(f'Processed zip: {PROCESSED_ZIP}')
print('\nUse /content/test_eval_data/processed for evaluation notebook.')